# 25 Lago LEAR Six-Year Benchmark

This notebook documents and runs a reproducible Lago et al. (2021)-style LEAR benchmark for Dutch hourly day-ahead prices.

- **Exact Lago-style D-only** benchmark uses a daily 247-feature structure.
- **D..D+4** benchmark is a no-leakage thesis adaptation with strict known-at filtering.


## 1. Title and Purpose

Goal: evaluate whether a transparent LEAR benchmark can approach or beat current complex FS2/FS3 stacks.


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists())
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.lago_lear_config import LagoLearBenchmarkConfig

config = LagoLearBenchmarkConfig()
config


## 2. Data Scope and Provenance

Inspect staged/official cleaned coverage, known-at assumptions, and missingness artifacts from run outputs.


## Importing and Constructing the Renewable Generation Forecast Variable

This benchmark imports ENTSO-E A69 day-ahead RES forecasts by PSR type for NL:
- `B16` solar
- `B18` wind offshore
- `B19` wind onshore

The aggregate Lago `x2` variable is constructed as:
`x2 = B16 + B18 + B19` per timestamp.

The exact D-only Lago variant uses only this aggregate `x2` vector (not separate component vectors) to preserve the 247-feature structure.

Known-at convention for A69 PSR imports:
- `known_at_rule = day_ahead_local_08_assumption_a69_psr`
- enforce `known_at_utc <= forecast_origin_utc` for feature eligibility.

Feature-ready RES policy used for Lago x2:
- strict hourly RES files are preserved as audit outputs
- benchmark x2 prefers feature-ready hourly files
- wind (`B18`,`B19`) is never zero-filled
- partial hours with `3/4` native points are accepted and flagged
- solar (`B16`) may be zero-filled only during physically dark NL hours (daylight classifier documented in cleaning diagnostics)
- aggregate x2 row is usable only when all three components are feature-ready for that hour

Missing RES forecast handling for the selected benchmark variant (`LEAR_LAGO_247_IMPUTED_X2`):
- remaining missing `x2` feature values are imputed with rolling **training-window median** only
- no row is dropped solely because `x2` has missing values when `--x2-missing-policy impute_training_median`
- `y_true` values are never imputed
- inspect: `features/missing_feature_summary.csv`, `features/imputation_summary.csv`, `features/dropped_rows_by_reason.csv`, `features/variant_feature_counts.csv`


## 3. Lago Feature Construction

Expected exact D-only feature count:
- 96 price lags
- 48 current exogenous vectors (24 load + 24 RES)
- 96 lagged exogenous vectors
- 7 weekday dummies
- **Total = 247**


## 4. D+4 Adaptation

D+4 uses direct lead-day/hour models with strict known-at filtering and explicit x2 policy metadata.


## 5. Model Methodology

For each calibration window (56, 84, 1092, 1456 days):
1. Standardize features
2. Select alpha via `LassoLarsIC(criterion='aic')`
3. Refit final coordinate-descent `Lasso`
4. Train per-hour (and per lead-day/hour for D+4)
5. Ensemble = arithmetic mean across available windows


## 6. Running the Pipeline

### Preflight / smoke
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_six_year_benchmark.py --smoke-test --max-origins 3 --allow-official-cleaned-fallback
```

### Import dry-run
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_data_import.py --dry-run --start-year 2019 --end-year 2025
```

### RES A69 by-PSR import dry-run
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_res_forecast_import.py --dry-run --start-year 2019 --end-year 2025 --psr-types B16 B18 B19
```

### Actual import
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_data_import.py --run-import --start-year 2019 --end-year 2025 --output-root data/00_raw_lago_lear_six_year
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_res_forecast_import.py --run-import --start-year 2019 --end-year 2025 --output-root data/00_raw_lago_lear_six_year --psr-types B16 B18 B19
```

### Cleaning dry-run
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_cleaning.py --dry-run
```

### Actual cleaning
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_cleaning.py --run-cleaning --raw-root data/00_raw_lago_lear_six_year --output-root data/01_cleaned_lago_lear_six_year
```

### Full D-only
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_six_year_benchmark.py --build-features --run-d-only --evaluate --plot-selected-weeks --compare-existing --split-policy lago_104w_test
```

### Full D+4
```bash
.venv\Scripts\python.exe scripts/Data/02_Forecasting/01_DA_prices/run_lago_lear_six_year_benchmark.py --build-features --run-dplus4 --evaluate --plot-selected-weeks --compare-existing --split-policy lago_104w_test --dplus4-x2-policy strict_no_future_x2
```


## 7. Evaluation Results
Load `metrics_overall.csv`, `metrics_by_lead_day.csv`, `operational_diagnostics.csv`, and `dm_tests.csv` from a run directory.


## 8. Selected-Week Visual Interpretation
Inspect winter/summer/high-volatility/high-price/low-price/negative-price weeks and ranking diagnostics.


## 9. Comparison with Current FS2/FS3
Use aligned timestamps/origins only. Mark non-overlapping periods as not comparable.


## 10. Limitations and Thesis Interpretation

- D-only is the closest Lago-style replication.
- D+4 is a no-leakage adaptation, not an exact Lago replication.
- Known-at assumptions are conservative internal project assumptions.
- MAPE is reported but not used for model selection under negative/near-zero prices.


## 11. Final Conclusion Template

- Does Lago LEAR ensemble beat naive?
- Does it approach FS3?
- Which window performs best?
- Is short-window or long-window calibration better?
- Does D+4 degrade smoothly with lead_day?
- Does top/bottom-hour identification support optimization needs?
